# Do-PFN for Marketing Mix Modeling

## Causal Effect Estimation as Alternative to Bayesian MMM (PyMC-Marketing)

This notebook implements a Do-PFN-based approach to estimating channel-level causal effects in a Marketing Mix context. The structure mirrors the Bayesian MMM, TabPFN, and CausalPFN notebooks to enable direct comparison where possible.

**Research Question:** Can Do-PFN — a pretrained transformer that predicts conditional interventional distributions (CIDs) from observational data — estimate meaningful channel-level treatment effects without any manually specified structural priors, and how do its estimates compare to the Bayesian MMM benchmark?

**Important Note on Comparability:** Like CausalPFN, Do-PFN operates in a fundamentally different paradigm from the Bayesian MMM and TabPFN.

| Model | Question answered | Output | Comparability |
|---|---|---|---|
| Bayesian MMM | How much revenue did each channel generate? | Channel contributions + ROAS | Reference model |
| TabPFN | What revenue is expected given these inputs? | Point predictions | Directly comparable via MAE/RMSE/MAPE |
| CausalPFN | What is the causal effect of advertising on revenue? | CATE estimates | Not directly comparable to MMM ROAS |
| Do-PFN | What is the interventional distribution p(y \| do(t), x)? | CID → CATE via subtraction | Not directly comparable to MMM ROAS |

**Key difference between Do-PFN and CausalPFN:**
Do-PFN does not estimate CATE directly. Instead, it predicts the full conditional interventional distribution (CID) p(y | do(t), x). CATE is then derived by subtracting the expected outcome under do(t=0) from the expected outcome under do(t=1). This makes Do-PFN more principled from a causal inference perspective, as it models the entire interventional distribution rather than only its mean.

**Model variants implemented:**

| Variant | Adstock Preprocessing | Treatment Definition |
|---|---|---|
| Do-PFN Raw | ✗ | Binary (spend > 0) |
| Do-PFN Adstock | ✓ (posterior means from Bayesian MMM) | Binary (adstocked spend > 0) |

## 1. Setup & Installation

Do-PFN is not available on PyPI. The repository must be cloned directly from GitHub and dependencies installed manually.

**Environment requirements:**
- Python 3.10 (Do-PFN is incompatible with Python 3.12+ due to internal PyTorch transformer import changes)
- PyTorch 2.1
- A dedicated conda environment (`dopfn_env`) is recommended

**Setup steps (run once in terminal before opening this notebook):**
```bash
conda create -n dopfn_env python=3.10 -y
conda activate dopfn_env
pip install torch==2.1.0 torchvision==0.16.0
git clone https://github.com/jr2021/Do-PFN.git
cd Do-PFN
pip install einops torchmetrics==1.2.0 pyreadr requests scikit-learn matplotlib seaborn tqdm numpy==1.26.4
```

The `DoPFNRegressor` class is located in `scripts/transformer_prediction_interface/base.py` within the cloned repository.

In [ ]:
import sys
import os

# Add the Do-PFN repository to the Python path
# Adjust this path to match your local clone of the Do-PFN repository
DOPFN_PATH = os.path.expanduser('~/Desktop/Seminararbeit/RobynData/Do-PFN')
sys.path.insert(0, DOPFN_PATH)

import torch
import numpy as np
import pandas as pd
import requests
import pyreadr
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
from sklearn.preprocessing import MaxAbsScaler, StandardScaler

warnings.filterwarnings('ignore')

# Import DoPFNRegressor from the local repository
from scripts.transformer_prediction_interface.base import DoPFNRegressor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('All imports successful!')

## 2. Data Loading

The same Robyn simulated weekly dataset used in the Bayesian MMM, TabPFN, and CausalPFN notebooks is loaded here. Using the identical dataset ensures that any differences in estimated channel effects are attributable to modeling choices rather than data differences.

The dataset contains 208 weekly observations with:
- **Target variable:** `revenue` — weekly revenue
- **Media channels:** TV, OOH, Print, Facebook, Search spend
- **Control variables:** competitor sales, newsletter subscribers

A chronological 80/20 split is applied — identical to all other notebooks. The chronological split preserves the temporal structure of the data and prevents the model from learning from future observations during training.

In [ ]:
# Download Robyn simulated dataset from GitHub
# Identical source as Bayesian MMM, TabPFN, and CausalPFN notebooks
url = 'https://github.com/facebookexperimental/Robyn/raw/main/R/data/dt_simulated_weekly.RData'
response = requests.get(url)
with open('dt_simulated_weekly.RData', 'wb') as f:
    f.write(response.content)

result = pyreadr.read_r('dt_simulated_weekly.RData')
df = result['dt_simulated_weekly']

df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)

target       = 'revenue'
media_cols   = ['tv_S', 'ooh_S', 'print_S', 'facebook_S', 'search_S']
control_cols = ['competitor_sales_B', 'newsletter']

# Chronological 80/20 split — identical to all other notebooks
split_idx = int(len(df) * 0.8)
train_df  = df.iloc[:split_idx].copy()
test_df   = df.iloc[split_idx:].copy()

print(f'Total observations : {len(df)}')
print(f'Train observations : {len(train_df)}')
print(f'Test observations  : {len(test_df)}')
df[['DATE', 'revenue'] + media_cols].head()

**Findings**

The Robyn simulated dataset was successfully loaded and split into training and test periods. The chronological split produces 166 training weeks and 42 test weeks, identical to the Bayesian MMM, TabPFN, and CausalPFN experiments. This ensures full comparability of evaluation conditions across all four modeling approaches.

## 3. Adstock Transformation

A geometric adstock transformation is applied as preprocessing before fitting Do-PFN. This gives the model access to temporal carryover information that it cannot learn on its own, since Do-PFN — like TabPFN and CausalPFN — treats observations as independent rows without temporal awareness.

The adstock alpha values are taken from the Bayesian MMM posterior means:

| Channel | Alpha | Decay speed |
|---|---|---|
| tv_S | 0.241 | Fast decay |
| ooh_S | 0.572 | Slow decay |
| print_S | 0.428 | Medium decay |
| facebook_S | 0.540 | Slow decay |
| search_S | 0.558 | Slow decay |

**Important:** The transformation is applied to the full dataset before splitting to correctly propagate carryover effects across the train-test boundary. If adstock were computed separately on train and test, the first weeks of the test period would miss the carryover contribution from the final weeks of the training period.

Do-PFN is run in two variants:
- **Raw:** raw media spend values without any adstock preprocessing
- **Adstock:** adstock-transformed spend values as covariates

In [ ]:
def geometric_adstock(x: np.ndarray, alpha: float, l_max: int = 8) -> np.ndarray:
    """
    Geometric adstock transformation.

    Parameters
    ----------
    x     : array of weekly media spend values
    alpha : decay parameter (0 = no carryover, 1 = no decay)
    l_max : maximum lag length in weeks (identical to Bayesian MMM: l_max=8)

    Returns
    -------
    x_adstocked : array of adstock-transformed spend values
    """
    x_adstocked = np.zeros_like(x, dtype=float)
    for t in range(len(x)):
        for lag in range(min(l_max, t + 1)):
            x_adstocked[t] += (alpha ** lag) * x[t - lag]
    return x_adstocked


# Posterior mean adstock alphas from the Bayesian MMM
adstock_alphas = {
    'tv_S':       0.241,
    'ooh_S':      0.572,
    'print_S':    0.428,
    'facebook_S': 0.540,
    'search_S':   0.558
}

# Apply adstock to the full dataset before splitting
df_adstock = df.copy()
for col, alpha in adstock_alphas.items():
    df_adstock[col + '_adstock'] = geometric_adstock(df[col].values, alpha=alpha)

adstock_cols  = [col + '_adstock' for col in media_cols]
train_adstock = df_adstock.iloc[:split_idx].copy()
test_adstock  = df_adstock.iloc[split_idx:].copy()

print('Adstock transformation complete.')
print('Adstock columns created:', adstock_cols)
df_adstock[['DATE'] + adstock_cols].head()

**Findings**

The geometric adstock transformation was successfully applied to all five media channels using the posterior mean alpha values estimated by the Bayesian MMM. The adstocked spend values accumulate delayed carryover effects across up to eight lagged weeks, consistent with the `l_max=8` specification used in the Bayesian MMM.

Channels with higher alpha values (OOH: 0.572, Search: 0.558, Facebook: 0.540) show stronger cumulative adstock values compared to TV (0.241), which decays more rapidly.

## 4. Scaling

Do-PFN — like the Bayesian MMM, TabPFN, and CausalPFN — benefits from normalized input features. We apply the same scaling pipeline as in the other notebooks:

- **Media channels:** `MaxAbsScaler` — scales values to [-1, 1] range while preserving sparsity
- **Control variables:** `StandardScaler` — zero mean, unit variance
- **Target variable:** kept on the original scale for interpretability

**Important:** The scalers are always fitted on training data only and then applied to test data. This prevents information leakage from the test set into the model.

Both the raw and the adstock-transformed feature sets are scaled independently, since the magnitude of adstock-transformed values differs from raw spend values.

In [ ]:
def scale_data(X_train, X_test, m_cols, c_cols):
    """
    Scale media columns with MaxAbsScaler and control columns with StandardScaler.
    Both scalers are fitted on training data only to prevent data leakage.
    """
    media_scaler   = MaxAbsScaler()
    control_scaler = StandardScaler()

    X_train = X_train.copy()
    X_test  = X_test.copy()

    X_train[m_cols] = media_scaler.fit_transform(X_train[m_cols])
    X_test[m_cols]  = media_scaler.transform(X_test[m_cols])

    X_train[c_cols] = control_scaler.fit_transform(X_train[c_cols])
    X_test[c_cols]  = control_scaler.transform(X_test[c_cols])

    return X_train, X_test


# Scale RAW feature sets
X_train_raw, X_test_raw = scale_data(
    train_df[media_cols + control_cols],
    test_df[media_cols + control_cols],
    media_cols, control_cols
)

# Scale ADSTOCK feature sets
X_train_ads, X_test_ads = scale_data(
    train_adstock[adstock_cols + control_cols],
    test_adstock[adstock_cols + control_cols],
    adstock_cols, control_cols
)

# Target on original scale
y_train = train_df[target].values.astype(np.float32)
y_test  = test_df[target].values.astype(np.float32)

print('Scaling complete.')
print(f'X_train_raw shape : {X_train_raw.shape}')
print(f'X_train_ads shape : {X_train_ads.shape}')

**Findings**

The scaling procedure successfully normalized both feature sets. Media channels are bounded within the [-1, 1] range after MaxAbs scaling, while control variables have been centered and standardized.

The sparsity pattern visible in the raw media features — with many values near zero indicating weeks without advertising activity — is preserved by the `MaxAbsScaler`, which does not shift the zero point.

## 5. Do-PFN: What It Does and How It Works

Do-PFN is a pretrained transformer model designed to predict **Conditional Interventional Distributions (CIDs)** from observational data — without any model training or hyperparameter tuning.

**Key concepts:**

- **Treatment (T):** A binary variable indicating whether advertising was active in a given week (spend > 0). Do-PFN expects the treatment as the **first column** of the feature matrix — this is the fundamental interface difference from CausalPFN.

- **Covariates (X):** All other media channels and control variables (columns 1 onwards in the feature matrix).

- **Outcome (Y):** Weekly revenue.

- **CID (Conditional Interventional Distribution):** Do-PFN predicts the full distribution p(y | do(t), x) — the distribution of outcomes under a forced intervention on the treatment, given observed covariates.

- **CATE estimation:** The Conditional Average Treatment Effect is derived from the CID by subtracting expected outcomes:
  CATE = E[Y | do(T=1), X] − E[Y | do(T=0), X]

**How Do-PFN differs from CausalPFN:**

| Aspect | CausalPFN | Do-PFN |
|---|---|---|
| Treatment input | Passed separately: `fit(X_cov, T, Y)` | First column of X: `fit(X_with_T_col0, Y)` |
| Primary output | CATE directly | Full CID: p(y \| do(t), x) |
| CATE estimation | `model.estimate_cate(X_test)` | `predict_full(x_t1)['mean'] - predict_full(x_t0)['mean']` |
| Causal assumption | Ignorability (unconfoundedness) | SCM-based; robust to unobserved confounding |
| Pre-training | Synthetic datasets with known causal structures | Synthetic SCMs; models distribution shift between observational and interventional data |

**Procedure for each channel:**
1. Build feature matrix X where column 0 = treatment T (binary), columns 1+ = covariates
2. Fit Do-PFN on (X_train, y_train)
3. Create two test matrices: x_t1 (T=1 everywhere) and x_t0 (T=0 everywhere)
4. CATE = predict_full(x_t1)['mean'] − predict_full(x_t0)['mean']

## 6. Variant 1 — Do-PFN Raw (No Adstock)

In this first variant, raw (unmodified) spend values are used as covariates. The treatment indicator is derived directly from the original spend values: a week is treated if any spend was recorded for that channel.

This is the most minimal setup and does not encode any marketing-specific structural knowledge. It tests whether Do-PFN can recover meaningful causal effects from raw observational marketing data without temporal preprocessing.

In [ ]:
results_raw = {}

for channel in media_cols:
    print(f'\nChannel: {channel}')

    # All other channels + controls are covariates
    other_cols     = [c for c in media_cols if c != channel]
    covariate_cols = other_cols + control_cols

    # Binary treatment: was this channel active this week?
    T_train = (train_df[channel].values > 0).astype(np.float32).reshape(-1, 1)
    T_test  = (test_df[channel].values > 0).astype(np.float32).reshape(-1, 1)

    # Covariate matrices (already scaled)
    X_cov_train = X_train_raw[covariate_cols].values.astype(np.float32)
    X_cov_test  = X_test_raw[covariate_cols].values.astype(np.float32)

    # Do-PFN expects treatment as the FIRST column of the feature matrix
    X_train_dopfn = np.hstack([T_train, X_cov_train])
    X_test_dopfn  = np.hstack([T_test,  X_cov_test])

    # Fit Do-PFN on training data
    dopfn = DoPFNRegressor()
    dopfn.fit(X_train_dopfn, y_train)

    # CATE: E[Y | do(T=1), X] - E[Y | do(T=0), X]
    x_t1 = deepcopy(X_test_dopfn); x_t1[:, 0] = 1.0  # intervene: set T=1
    x_t0 = deepcopy(X_test_dopfn); x_t0[:, 0] = 0.0  # intervene: set T=0

    y_t1 = dopfn.predict_full(torch.tensor(x_t1))['mean']  # E[Y | do(T=1), X]
    y_t0 = dopfn.predict_full(torch.tensor(x_t0))['mean']  # E[Y | do(T=0), X]
    cate  = np.array(y_t1) - np.array(y_t0)

    results_raw[channel] = cate
    print(f'  Mean CATE (Raw): {np.mean(cate):>12,.2f}')
    print(f'  Std  CATE (Raw): {np.std(cate):>12,.2f}')

print('\nDo-PFN Raw complete.')

In [ ]:
fig, axes = plt.subplots(1, len(media_cols), figsize=(18, 4), sharey=False)

for ax, channel in zip(axes, media_cols):
    ax.hist(results_raw[channel], bins=20, edgecolor='black', color='steelblue', alpha=0.8)
    ax.axvline(np.mean(results_raw[channel]), color='red',
               linestyle='--', linewidth=1.5, label=f"Mean: {np.mean(results_raw[channel]):,.0f}")
    ax.axvline(0, color='black', linestyle='-', linewidth=0.8)
    ax.set_title(channel, fontsize=10)
    ax.set_xlabel('CATE')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)

plt.suptitle('CATE Distribution per Channel — Do-PFN Raw', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

**Findings: Do-PFN Raw**

Do-PFN estimated a conditional average treatment effect for each media channel across all 42 test period observations by predicting the full conditional interventional distribution and deriving CATE from the difference in expected outcomes under do(T=1) and do(T=0).

The CATE distributions show variation across weeks, reflecting the model's ability to condition its causal estimates on the observed context — that is, the effects are not constant but change depending on which other channels were active and what the control variable values were in each week.

Channels with consistently positive mean CATE values indicate that the model estimates a positive causal effect of advertising on revenue — directionally consistent with the Bayesian MMM. The Do-PFN Raw variant achieves stronger directional agreement with the Bayesian MMM than CausalPFN Raw, suggesting that Do-PFN's SCM-based pretraining may provide better causal identification in observational marketing data.

## 7. Variant 2 — Do-PFN Adstock

In this second variant, the adstock-transformed spend values are used as covariates. The treatment indicator is now derived from the adstocked values: a week is treated if the cumulative adstock signal for that channel is positive.

By incorporating adstock-transformed features, the model has access to temporal carryover information that Do-PFN cannot learn on its own — since it treats each row as an independent observation. This variant tests whether encoding lagged advertising persistence changes the estimated causal effects.

The adstock alpha values are identical to those used in the Bayesian MMM and CausalPFN notebooks, enabling a consistent comparison.

In [ ]:
results_ads = {}

for channel in media_cols:
    print(f'\nChannel: {channel}')

    ads_col        = channel + '_adstock'
    other_ads      = [c + '_adstock' for c in media_cols if c != channel]
    covariate_cols = other_ads + control_cols

    # Binary treatment on adstock basis: was there any cumulative signal this week?
    T_train_ads = (train_adstock[ads_col].values > 0).astype(np.float32).reshape(-1, 1)
    T_test_ads  = (test_adstock[ads_col].values > 0).astype(np.float32).reshape(-1, 1)

    X_cov_train = X_train_ads[covariate_cols].values.astype(np.float32)
    X_cov_test  = X_test_ads[covariate_cols].values.astype(np.float32)

    # Do-PFN expects treatment as the FIRST column
    X_train_dopfn = np.hstack([T_train_ads, X_cov_train])
    X_test_dopfn  = np.hstack([T_test_ads,  X_cov_test])

    dopfn_ads = DoPFNRegressor()
    dopfn_ads.fit(X_train_dopfn, y_train)

    x_t1 = deepcopy(X_test_dopfn); x_t1[:, 0] = 1.0
    x_t0 = deepcopy(X_test_dopfn); x_t0[:, 0] = 0.0

    y_t1 = dopfn_ads.predict_full(torch.tensor(x_t1))['mean']
    y_t0 = dopfn_ads.predict_full(torch.tensor(x_t0))['mean']
    cate  = np.array(y_t1) - np.array(y_t0)

    results_ads[channel] = cate
    print(f'  Mean CATE (Adstock): {np.mean(cate):>12,.2f}')
    print(f'  Std  CATE (Adstock): {np.std(cate):>12,.2f}')

print('\nDo-PFN Adstock complete.')

In [ ]:
fig, axes = plt.subplots(1, len(media_cols), figsize=(18, 4), sharey=False)

for ax, channel in zip(axes, media_cols):
    ax.hist(results_ads[channel], bins=20, edgecolor='black', color='darkorange', alpha=0.8)
    ax.axvline(np.mean(results_ads[channel]), color='red',
               linestyle='--', linewidth=1.5, label=f"Mean: {np.mean(results_ads[channel]):,.0f}")
    ax.axvline(0, color='black', linestyle='-', linewidth=0.8)
    ax.set_title(channel, fontsize=10)
    ax.set_xlabel('CATE')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)

plt.suptitle('CATE Distribution per Channel — Do-PFN Adstock', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

**Findings: Do-PFN Adstock**

The adstock-informed variant of Do-PFN produced CATE estimates that incorporate the temporal carryover structure of advertising effects.

Comparing the adstock and raw variants allows assessment of whether encoding lagged advertising persistence changes the estimated causal effects. The adstock transformation changes the treatment signal: in the raw variant, T=1 only when spend was recorded in that specific week; in the adstock variant, T=1 whenever there is any residual carryover from recent advertising activity. This means the treatment is active more often in the adstock variant, which may affect the estimated effect magnitude.

## 8. Mean CATE Summary & Sign Agreement with Bayesian MMM

The mean CATE across all 42 test period observations summarizes the estimated causal effect of each channel — averaged over the observed variation in contexts (other channels, controls).

A positive mean CATE indicates that the model estimates advertising in that channel to causally increase revenue on average. The sign pattern provides a directional comparison with the Bayesian MMM, which estimates positive contributions for all channels.

**Sign agreement is used as the primary comparison metric** because the magnitudes of CATE and MMM channel contributions are not directly comparable — a limitation discussed in detail in the following section.

In [ ]:
# Mean CATE summary table
cate_summary = pd.DataFrame({
    'Mean CATE (Raw)':     {ch: np.mean(results_raw[ch]) for ch in media_cols},
    'Mean CATE (Adstock)': {ch: np.mean(results_ads[ch]) for ch in media_cols},
}).round(2)

print('=' * 60)
print('Mean CATE per Channel — Do-PFN')
print('=' * 60)
print(cate_summary.to_string())

# Sign agreement with Bayesian MMM (all channels positive)
sign_df = pd.DataFrame({
    'Bayesian MMM':    ['+'] * len(media_cols),
    'Do-PFN Raw':     ['+' if np.mean(results_raw[ch]) > 0 else '-' for ch in media_cols],
    'Do-PFN Adstock': ['+' if np.mean(results_ads[ch]) > 0 else '-' for ch in media_cols],
}, index=media_cols)

print('\n' + '=' * 60)
print('Sign Agreement with Bayesian MMM (all channels positive)')
print('=' * 60)
print(sign_df.to_string())

match_raw = sum(np.mean(results_raw[ch]) > 0 for ch in media_cols) / len(media_cols) * 100
match_ads = sum(np.mean(results_ads[ch]) > 0 for ch in media_cols) / len(media_cols) * 100
print(f'\nDo-PFN Raw     : {match_raw:.0f}% sign agreement')
print(f'Do-PFN Adstock : {match_ads:.0f}% sign agreement')
print(f'Bayesian MMM   : 100% (reference)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, results, color_pos, color_neg) in zip(axes, [
    ('Do-PFN Raw',     results_raw, 'steelblue',  'salmon'),
    ('Do-PFN Adstock', results_ads, 'darkorange', 'salmon'),
]):
    means  = [np.mean(results[ch]) for ch in media_cols]
    colors = [color_pos if v > 0 else color_neg for v in means]

    bars = ax.bar(media_cols, means, color=colors, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Marketing Channel')
    ax.set_ylabel('Mean CATE')
    ax.tick_params(axis='x', rotation=30)

    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(abs(v) for v in means) * 0.01,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Mean CATE per Channel — Blue/Orange = Positive, Red = Negative', fontsize=11)
plt.tight_layout()
plt.show()

## 9. Why Do-PFN and Bayesian MMM Cannot Be Directly Compared

A natural ambition in this comparison is to place Do-PFN CATE values side-by-side with Bayesian MMM ROAS estimates. However, this comparison is fundamentally misleading — not because of implementation differences, but because the two models answer structurally different questions.

**What the Bayesian MMM measures**

The Bayesian MMM decomposes total observed revenue into contributions attributed to each marketing channel over the training period. ROAS is defined as:

ROAS_MMM = channel_contribution_total / channel_spend_total

The channel contribution is the model's estimate of how much revenue would not have been generated if that channel had not spent at all — computed by integrating the posterior distribution of the channel's contribution over the entire training period. This is a **continuous quantity** that reflects both whether advertising occurred and how much was spent.

**What Do-PFN measures**

Do-PFN estimates the Conditional Average Treatment Effect (CATE) derived from the conditional interventional distribution p(y | do(t), x). The treatment is binary:

- T = 1 if spend > 0 (advertising was active this week)
- T = 0 if spend = 0 (no advertising this week)

The CATE therefore estimates: *"By how much does revenue increase in a week where advertising ran compared to a week with no advertising at all?"*

This is a qualitatively different question from what the Bayesian MMM answers.

**Three reasons the numbers are not comparable**

1. **Continuous vs. binary treatment:** The Bayesian MMM uses actual continuous spend values. A channel spending €500k and one spending €5k generate different contribution estimates. Do-PFN collapses both to T=1 — the intensity of spend is invisible to the model.

2. **Different time aggregations:** The Bayesian MMM ROAS is computed over the entire training period (166 weeks). The Do-PFN CATE is estimated on the test period (42 weeks).

3. **Different causal assumptions:** The Bayesian MMM operates under strong structural assumptions encoded in its priors. Do-PFN makes different assumptions derived from pretraining on synthetic SCMs. Even if both find a "positive effect" for TV, the underlying causal quantity is not the same.

**What can legitimately be compared**

| Comparison | Legitimate? | Reason |
|---|---|---|
| CATE sign vs. MMM contribution sign | ✓ Yes | Both indicate direction of effect |
| Channel ranking by CATE vs. by ROAS | Partially | Rankings may differ due to binary treatment |
| MMM MAPE vs. Do-PFN predictions | ✗ No | Do-PFN does not produce revenue point predictions |
| MMM ROAS magnitude vs. Do-PFN CATE | ✗ No | Numerators measure structurally different quantities |

## 10. Full Model Comparison: Do-PFN vs. CausalPFN vs. TabPFN vs. Bayesian MMM

This section consolidates results from all four modeling approaches into a unified comparison.

**Important note on output types:**
- **Bayesian MMM & TabPFN** produce revenue predictions → comparable via MAE, RMSE, MAPE
- **CausalPFN & Do-PFN** produce CATE estimates → only comparable via sign agreement and channel ranking

The sign agreement heatmap visualizes which models agree with the Bayesian MMM on the direction of each channel's effect.

In [ ]:
# ── Results from all notebooks ────────────────────────────────────

# Bayesian MMM ROAS (reference)
bayesian_roas = {
    'tv_S': 0.261, 'ooh_S': 0.105, 'print_S': 0.080,
    'facebook_S': 0.126, 'search_S': 0.048
}

# TabPFN ROAS (from TabPFN notebook)
tabpfn_raw_roas = {
    'tv_S': 4.550, 'ooh_S': 0.148, 'print_S': 5.582,
    'facebook_S': 8.098, 'search_S': 0.429
}
tabpfn_ads_roas = {
    'tv_S': 5.654, 'ooh_S': 0.320, 'print_S': 10.959,
    'facebook_S': 15.657, 'search_S': 0.118
}
tabpfn_sat_roas = {
    'tv_S': 9.239, 'ooh_S': 0.402, 'print_S': 14.975,
    'facebook_S': 16.511, 'search_S': -5.104
}

# CausalPFN Mean CATE (from CausalPFN notebook)
causal_raw = {
    'tv_S': 381131.03, 'ooh_S': -65830.89, 'print_S': -46365.78,
    'facebook_S': -3943.10, 'search_S': -158678.73
}
causal_ads = {
    'tv_S': 908742.69, 'ooh_S': -981255.31, 'print_S': 725554.06,
    'facebook_S': -354230.81, 'search_S': -1145777.88
}

# Do-PFN Mean CATE (from this notebook)
dopfn_raw = {ch: np.mean(results_raw[ch]) for ch in media_cols}
dopfn_ads = {ch: np.mean(results_ads[ch]) for ch in media_cols}

# ── Sign Agreement Matrix ─────────────────────────────────────────
sign_matrix = pd.DataFrame({
    'Bayesian MMM':         {ch: 1 for ch in media_cols},
    'TabPFN (raw)':         {ch: 1 if tabpfn_raw_roas[ch] > 0 else 0 for ch in media_cols},
    'TabPFN (adstock)':     {ch: 1 if tabpfn_ads_roas[ch] > 0 else 0 for ch in media_cols},
    'TabPFN (adstock+sat)': {ch: 1 if tabpfn_sat_roas[ch] > 0 else 0 for ch in media_cols},
    'CausalPFN (raw)':      {ch: 1 if causal_raw[ch] > 0 else 0 for ch in media_cols},
    'CausalPFN (adstock)':  {ch: 1 if causal_ads[ch] > 0 else 0 for ch in media_cols},
    'Do-PFN (raw)':         {ch: 1 if dopfn_raw[ch] > 0 else 0 for ch in media_cols},
    'Do-PFN (adstock)':     {ch: 1 if dopfn_ads[ch] > 0 else 0 for ch in media_cols},
})

# ── Sign Agreement Score ──────────────────────────────────────────
print('=' * 60)
print('Sign Agreement with Bayesian MMM (all channels positive)')
print('=' * 60)
for model in sign_matrix.columns[1:]:
    score = sign_matrix[model].sum()
    print(f'  {model:<28}: {score}/{len(media_cols)} ({score/len(media_cols)*100:.0f}%)')
print(f'  {"Bayesian MMM":<28}: {len(media_cols)}/{len(media_cols)} (100%) — reference')

# ── Sign Agreement Heatmap ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    sign_matrix.T,
    annot=True, fmt='d',
    cmap=['#e74c3c', '#2ecc71'],
    linewidths=0.5, linecolor='white',
    cbar=False, ax=ax,
    vmin=0, vmax=1
)
ax.set_title('Sign Agreement Heatmap — 1 = Positive Effect, 0 = Negative Effect', fontsize=12)
ax.set_xlabel('Marketing Channel')
ax.set_ylabel('Model')
plt.tight_layout()
plt.show()

# ── Forecast Accuracy (Bayesian MMM and TabPFN only) ──────────────
print('\n' + '=' * 60)
print('Forecast Accuracy — Test Period')
print('Note: CausalPFN and Do-PFN estimate CATE, not revenue')
print('      and therefore cannot be evaluated with MAE/RMSE/MAPE')
print('=' * 60)
metrics = pd.DataFrame({
    'Model': ['Bayesian MMM', 'TabPFN (raw)', 'TabPFN (adstock)', 'TabPFN (adstock+sat)'],
    'MAE':   [115134.78, 99086.33, 94207.08, 97901.44],
    'RMSE':  [218368.30, 214200.44, 219723.31, 222992.17],
    'MAPE':  [8.53, 7.97, 7.41, 7.68],
}).set_index('Model')
print(metrics.round(2).to_string())

## 11. Do-PFN Summary

Do-PFN was applied to the Robyn simulated weekly dataset to estimate channel-level conditional average treatment effects (CATE) as an alternative to the Bayesian MMM's channel contribution analysis.

Two model variants were implemented:
- **Do-PFN Raw:** no adstock preprocessing, binary treatment on raw spend
- **Do-PFN Adstock:** geometric adstock preprocessing using Bayesian MMM posterior mean alpha values

For each channel, the model was fitted on the training period and CATE estimates were generated for all 42 test period observations by predicting the conditional interventional distribution and deriving CATE as the difference in expected outcomes under do(T=1) and do(T=0).

**Sign agreement with Bayesian MMM:**

| Model | Channels with correct sign | Agreement rate |
|---|---|---|
| CausalPFN Raw | tv_S only | 20% |
| CausalPFN Adstock | tv_S, print_S | 40% |
| Do-PFN Raw | tv_S, ooh_S, print_S, facebook_S | 80% |
| Do-PFN Adstock | tv_S, print_S | 40% |
| Bayesian MMM | all | 100% (reference) |

**Interpretation:**

Do-PFN Raw achieves substantially stronger directional agreement with the Bayesian MMM (80%) compared to CausalPFN Raw (20%). This suggests that Do-PFN's SCM-based pretraining — which explicitly models the distribution shift between observational and interventional data — provides better causal identification in this observational marketing setting.

However, the adstock variant shows weaker agreement (40%), suggesting that the adstock transformation changes the treatment signal in a way that complicates causal identification: when the treatment is derived from cumulative adstock rather than raw spend, the boundary between treated and untreated weeks becomes less sharp, which may interfere with Do-PFN's learned causal adjustment mechanisms.

**Critical limitation:**

The ROAS magnitudes of the Bayesian MMM and the CATE values of Do-PFN are not directly comparable. Do-PFN uses a binary treatment variable (advertising active or not), while the Bayesian MMM uses continuous spend values processed through adstock and saturation transformations. The CATE therefore measures a qualitatively different quantity from the MMM's channel contribution — a direct numerical comparison would be misleading.

The appropriate role of Do-PFN in this comparison is as a **directional causal check** — estimating whether advertising causally increases revenue at all, without requiring the structural assumptions that the Bayesian MMM depends on. In this experiment, Do-PFN Raw achieves strong directional agreement with the Bayesian MMM, providing meaningful complementary evidence that most channels have a genuine causal effect on revenue.